# UI Explore Playground

Подбор локаторов для autoUI: императивный обход pywinauto и проверка `Locator` pipeline.

Требования: Windows, открытое целевое приложение (по умолчанию Notepad++).

In [1]:
import os

os.chdir("..")
os.getcwd()

'd:\\D\\Work\\Dev\\autoUI\\src'

In [2]:
# %load_ext autoreload
# %autoreload 2

In [3]:
import pywinauto

In [4]:
from autoui.explore import (
    ExplorerSession,
    filter_children,
    filter_descendants,
    find_desktop_windows,
    list_menu_items,
    path_from_root,
)
from autoui.locators import ChildOp, FindDescendantsOp, FilterOp, Locator, TakeOp

In [6]:
WINDOW_TITLE = "Notepad++"

session = ExplorerSession(WINDOW_TITLE, verbose_locators=True)
window = session.connect(index=0)
window

<uiawrapper.UIAWrapper - '*новый 3 - Notepad++', Dialog, -3099070881811220391>

In [7]:
session.window

<uiawrapper.UIAWrapper - '*новый 3 - Notepad++', Dialog, -3099070881811220391>

## Императивный поиск

Меняйте индексы и фильтры; `session.list_indexed(...)` помогает подобрать номера.

In [8]:
# --- меняйте запрос ---
WINDOW_INDEX = 0
CHILD_INDEX = 6
DESC_FILTERS = {"control_type": "Button"}
CANDIDATE_INDEX = 22

# windows = find_desktop_windows(WINDOW_TITLE)
# window = windows[WINDOW_INDEX]
areas = session.window.children()
toolbar = areas[CHILD_INDEX]
candidates = toolbar.descendants(**DESC_FILTERS)
target = candidates[CANDIDATE_INDEX]

session.highlight(target, colour="green")
session.describe(target)
target

ElementProperties(name=None, automation_id=None, class_name=None, control_type='Button', enabled=True, visible=True)
#32769 | Рабочий стол 1
Notepad++ | *новый 3 - Notepad++
ReBarWindow32 | <no title>
ToolbarWindow32 | <no title>
<no class> | <no title>


<uia_controls.ButtonWrapper - '', Button, 6977815300220982273>

## Locator pipeline

`query_locator()` — все совпадения после pipeline (без `Take` в конце), как `toolbar.descendants(...)`.
`try_locator()` — один элемент через `driver.resolve` (как в UIMap); при N>1 без `Take` вернётся первый с предупреждением в trace/логе.

См. README — подраздел **Query vs resolve**.

In [9]:
# все кандидаты (без Take)
LOCATOR_QUERY = Locator([
    ChildOp(CHILD_INDEX),
    FindDescendantsOp(where=DESC_FILTERS),
])
candidates = session.query_locator(LOCATOR_QUERY)
session.list_indexed(candidates)

# один элемент (как в сценарии)
LOCATOR = Locator([
    ChildOp(CHILD_INDEX),
    FindDescendantsOp(where=DESC_FILTERS),
    TakeOp(CANDIDATE_INDEX),
])
element = session.try_locator(LOCATOR)
# element.control.draw_outline()  # OK, too
session.highlight(element.control, colour="green")
element

Locator pipeline (3 ops), success:
  [0] child(index=6)  in=1  out=1  ok
  [1] find_descendants({'control_type': 'Button'})  in=1  out=65  ok
  [2] take(index=22)  in=65  out=1  ok


PywinautoElement(control=<uia_controls.ButtonWrapper - '', Button, 6977815300220982273>)

In [10]:
# --- меняйте запрос ---
CHILD_INDEX = 6

# windows = find_desktop_windows(WINDOW_TITLE)
# window = windows[WINDOW_INDEX]
areas = window.children()
print('areas count: ', len(areas))
toolbar = areas[CHILD_INDEX]

target = toolbar

# DESC_FILTERS = {"control_type": "Button"}
# candidates = toolbar.descendants(**DESC_FILTERS)

# CANDIDATE_INDEX = 22
# target = candidates[CANDIDATE_INDEX]

session.highlight(target, colour="green")
print(path_from_root(target))

areas count:  9
#32769 | Рабочий стол 1
Notepad++ | *новый 3 - Notepad++
ReBarWindow32 | <no title>


In [11]:
# --- меняйте запрос ---
# CHILD_INDEX = 6

# windows = find_desktop_windows(WINDOW_TITLE)
# window = windows[WINDOW_INDEX]
# areas = window.children()
# toolbar = areas[CHILD_INDEX]

# target = toolbar

DESC_FILTERS = {"control_type": "Button"}
candidates = toolbar.descendants(**DESC_FILTERS)
print('candidates count: ', len(candidates))

CANDIDATE_INDEX = 22
target = candidates[CANDIDATE_INDEX]

session.highlight(target, colour="green")
session.describe(target)

candidates count:  65
ElementProperties(name=None, automation_id=None, class_name=None, control_type='Button', enabled=True, visible=True)
#32769 | Рабочий стол 1
Notepad++ | *новый 3 - Notepad++
ReBarWindow32 | <no title>
ToolbarWindow32 | <no title>
<no class> | <no title>


In [12]:
# target.click_input()  # С перемещением мыши на кнопку.
target.click()  # Без перемещения мыши на кнопку, быстрее.

<uia_controls.ButtonWrapper - '', Button, 6977815300220982273>

In [13]:
# Список детей / кандидатов с индексами
session.list_indexed(areas)
# session.list_indexed(candidates)

[0] Pane | 'You are an experienced instructor who evaluates knowledge of operator precedence and order of evaluation in C/C++ expressions.\r\nHere is how we work:\r\n1. I will send you a C/C++ expression \r\n   (for example: old = ptr->arr[i].value).\r\n2. You will analyze the expression and determine the exact order of operator execution according to the rules of C/C++, \r\n   but DO NOT show me the order.\r\n   hello!\r\n3. Then, I will write step by step which operator I press (for example: "[]", "->", ".", "=", "*", "+", etc.).\r\nIf I send "pressed operator [" or "pressed operator ]", treat it as "pressed the indexing operator".\r\n4. After each step, you will check my choice:\r\nIf I have chosen the correct operator, you will write:\r\n"Correct" and wait for the next input.\r\nIf I have chosen incorrectly, you will write:\r\n"Incorrect" and provide an explanation.\r\nhello\r\n\t\r\n\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t   \r\n\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t  \r\n\t\t\t\t\t\t

## Locator pipeline

Проверка через `PywinautoDriver.resolve`; при ошибке — trace в stdout.

In [14]:
LOCATOR = Locator([
    ChildOp(CHILD_INDEX),
    FindDescendantsOp(where=DESC_FILTERS),
    TakeOp(CANDIDATE_INDEX),
])

element = session.try_locator(LOCATOR)
element

Locator pipeline (2 ops), failed at step 1:
  [0] child(index=6)  in=1  out=1  ok
  [1] find_descendants({'control_type': 'Button'})  in=1  out=0  empty  ← FAILED
  reason: empty_set


In [15]:
# Shorthand: Locator.find(control_type="MenuBar")
menu_locator = Locator([
    FindDescendantsOp(where={"control_type": "MenuBar"}),
    TakeOp(0),
    FindDescendantsOp(where={"name": "Search"}),
    TakeOp(0),
])
session.try_locator(menu_locator)

Locator pipeline (3 ops), failed at step 2:
  [0] find_descendants({'control_type': 'MenuBar'})  in=1  out=2  ok
  [1] take(index=0)  in=2  out=1  ok
  [2] find_descendants({'name': 'Search'})  in=1  out=0  empty  ← FAILED
  reason: empty_set


## Меню (опционально)

In [16]:
list_menu_items(session.window.menu())

AttributeError: 'UIAWrapper' object has no attribute 'menu'

In [ ]:
session.clear_highlight()

In [ ]:
WINDOW_TITLE = "Конфигуратор"

session = ExplorerSession(WINDOW_TITLE, verbose_locators=True)
window = session.connect(index=0)
window

<uiawrapper.UIAWrapper - 'Конфигуратор ПО 'Лаборатории ММИС'', Dialog, -4846703788171400998>

In [ ]:
from autoui.explore import filter_children, filter_descendants


In [ ]:
# --- меняйте запрос ---
CHILD_INDEX = 0

# loc = Locator.find(automation_id="ucAdmin")
# res = session.try_locator(loc)

candidates = filter_children(window, automation_id="ucAdmin")
# print(len(candidates), "-", candidates)
# print()
pane_main = candidates[0]

candidates = filter_children(pane_main, automation_id="splitAdm")
# print(candidates)
pane_main2 = candidates[0]
target = pane_main2

areas = pane_main2.children()
target = areas[0]

root_navigation = areas[0]
root_workspace = areas[1]

root_workspace2 = root_workspace.children()[0].children()[0].children()[0]
plans_list_with_year_selector = root_workspace.children()[0].children()[0].children()[0]

plans_list = root_workspace2.children()[0].children()[0].children()[0]#.children()[0]

areas = plans_list.children()
print("areas count:", len(areas))
print(areas)
target = areas[0]

print()

session.highlight(target, colour="green")
session.describe(target)

areas count: 1
[<uia_controls.ListViewWrapper - '', Table, -8385602216069822364>]

ElementProperties(name=None, automation_id='gcPlan', class_name='WindowsForms10.Window.8.app.0.13965fa_r8_ad1', control_type='Table', enabled=True, visible=True)
#32769 | Рабочий стол 1
WindowsForms10.Window.8.app.0.13965fa_r8_ad1 | Конфигуратор ПО 'Лаборатории ММИС'
WindowsForms10.Window.8.app.0.13965fa_r8_ad1 | <no title>
WindowsForms10.Window.8.app.0.13965fa_r8_ad1 | splitAdm
WindowsForms10.Window.8.app.0.13965fa_r8_ad1 | <no title>
WindowsForms10.Window.8.app.0.13965fa_r8_ad1 | <no title>
WindowsForms10.Window.8.app.0.13965fa_r8_ad1 | <no title>
WindowsForms10.Window.8.app.0.13965fa_r8_ad1 | Правила переноса рабочих программ
WindowsForms10.Window.8.app.0.13965fa_r8_ad1 | Правила переноса рабочих программ
WindowsForms10.Window.8.app.0.13965fa_r8_ad1 | <no title>
WindowsForms10.Window.8.app.0.13965fa_r8_ad1 | <no title>
WindowsForms10.Window.8.app.0.13965fa_r8_ad1 | <no title>


LocatorOp = ChildOp | FindDescendantsOp | FilterOp | TakeOp

In [8]:
WINDOW_TITLE = "Cursor Agent"

session = ExplorerSession(WINDOW_TITLE, verbose_locators=True)
window = session.connect(index=0)
window

<uiawrapper.UIAWrapper - 'Cursor Agents', Dialog, 5779535376349804967>

In [9]:
LOCATOR = Locator([
    ChildOp(1),
    ChildOp(0),
    ChildOp(0),
    ChildOp(1),
    ChildOp(0),
    ChildOp(0),
    ChildOp(0),
    ChildOp(0),
    ChildOp(0),
    ChildOp(0),  # начало разделения по окну
    ChildOp(5),  # вся левая навигационная панель
    ChildOp(7),  # Workspaces pane
    ChildOp(0),
])

element = session.try_locator(LOCATOR)

if element:
    print('element\'s children count =', element.control.control_count())
    for i, ch in enumerate(element.control.children()):
        # ch.draw_outline('blue', thickness=6)
        session.highlight(ch, colour="blue", thickness=6)
        print('* child', i)
        print(ch, ch.get_properties())


    print(" == result ==")
    # element.control.draw_outline()  # OK, too
    session.highlight(element.control, colour="green", thickness=2)
    print(element, element.control.get_properties())
    # session.describe(element.control)

element

Locator pipeline (13 ops), success:
  [0] child(index=1)  in=1  out=1  ok
  [1] child(index=0)  in=1  out=1  ok
  [2] child(index=0)  in=1  out=1  ok
  [3] child(index=1)  in=1  out=1  ok
  [4] child(index=0)  in=1  out=1  ok
  [5] child(index=0)  in=1  out=1  ok
  [6] child(index=0)  in=1  out=1  ok
  [7] child(index=0)  in=1  out=1  ok
  [8] child(index=0)  in=1  out=1  ok
  [9] child(index=0)  in=1  out=1  ok
  [10] child(index=5)  in=1  out=1  ok
  [11] child(index=7)  in=1  out=1  ok
  [12] child(index=0)  in=1  out=1  ok
element's children count = 3
* child 0
uia_controls.ButtonWrapper - 'Search Agents', Button {'class_name': 'ui-icon-button ui-jyslct ui-3nfvp2 ui-6s0dn4 ui-l56j7k ui-2lah0s ui-9f619 ui-exx8yu ui-1xpa7k ui-18d9i69 ui-1uhho1l ui-dj266r ui-1yf7rl7 ui-at24cr ui-j3b58b ui-c342km ui-ng3xce ui-jb2p0i ui-1qlqyl8 ui-1pd3egz ui-15bjb6t ui-1ypdohk ui-1s07b3s ui-ijokvz ui-1k57tk5 ui-784prv ui-1t137rt ui-9v5kkp ui-1uczgqu ui-1725o6r ui-1wfwxd8 ui-7s97pk ui-67bb7w ui-aqnwrm ui

PywinautoElement(control=<uia_controls.ButtonWrapper - 'Workspaces Search Agents Customize Sidebar Open Workspace', Button, -5371252494747126076>)

In [10]:
element.click(internally=False)

In [11]:
LOCATOR = Locator([
    ChildOp(1),
    ChildOp(0),
    ChildOp(0),
    ChildOp(1),
    ChildOp(0),
    ChildOp(0),
    ChildOp(0),
    ChildOp(0),
    ChildOp(0),
    ChildOp(0),  # начало разделения по окну
    # ChildOp(5),  # вся левая навигационная панель
    # ChildOp(7),  # Workspaces pane
    # ChildOp(0),
    # FindDescendantsOp(where={"control_type": "Button"}),
    # TakeOp(0),
])

element = session.try_locator(LOCATOR)

if element:
    print('element\'s children count =', element.control.control_count())
    # List children
    # for i, ch in enumerate(element.control.children()):
    #     # ch.draw_outline('blue', thickness=6)
    #     session.highlight(ch, colour="blue", thickness=6)
    #     print('* child', i)
    #     print(ch, ch.get_properties())

    print(" == result ==")
    # element.control.draw_outline()  # OK, too
    session.highlight(element.control, colour="green", thickness=2)
    print(element, element.control.get_properties())
    session.describe(element.control)

element

Locator pipeline (10 ops), success:
  [0] child(index=1)  in=1  out=1  ok
  [1] child(index=0)  in=1  out=1  ok
  [2] child(index=0)  in=1  out=1  ok
  [3] child(index=1)  in=1  out=1  ok
  [4] child(index=0)  in=1  out=1  ok
  [5] child(index=0)  in=1  out=1  ok
  [6] child(index=0)  in=1  out=1  ok
  [7] child(index=0)  in=1  out=1  ok
  [8] child(index=0)  in=1  out=1  ok
  [9] child(index=0)  in=1  out=1  ok
element's children count = 13
 == result ==
PywinautoElement(control=<uiawrapper.UIAWrapper - '', GroupBox, 8889198004765212908>) {'class_name': 'flex flex-col size-full absolute inset-0 outline-none', 'friendly_class_name': 'GroupBox', 'texts': [''], 'control_id': None, 'rectangle': <RECT L0, T0, R1536, B1824>, 'is_visible': True, 'is_enabled': True, 'control_count': 13, 'is_keyboard_focusable': False, 'has_keyboard_focus': False, 'automation_id': ''}
ElementProperties(name=None, automation_id=None, class_name='flex flex-col size-full absolute inset-0 outline-none', control_ty

PywinautoElement(control=<uiawrapper.UIAWrapper - '', GroupBox, 8889198004765212908>)

In [30]:
LOCATOR = Locator([
    FindDescendantsOp(where={'automation_id': 'RootWebArea'}, depth=8, limit=1),  # без limit Это долго!!
    # TakeOp(0),  # можно опустить, если в пред. результат один
            # ChildOp(1),
            # ChildOp(0),
            # ChildOp(0),
            # ChildOp(1),
            # ChildOp(0),
            # ChildOp(0),
            # ChildOp(0),
            # ChildOp(0),

    ChildOp(0),
    ChildOp(0),  # начало разделения по окну
    FindDescendantsOp(where={"control_type": "Button"}, depth=2, limit=100),  # direct children only
    TakeOp(9),
    # ChildOp(5),  # вся левая навигационная панель
    # ChildOp(7),  # Workspaces pane
    # ChildOp(0),
])

element = session.try_locator(LOCATOR)

if 1 and element:
    print('element\'s children count =', element.control.control_count())
    # List children
    for i, ch in enumerate(element.control.children()):
        # ch.draw_outline('blue', thickness=6)
        session.highlight(ch, colour="blue", thickness=6)
        print('* child', i)
        print(ch, ch.get_properties())

    print(" == result ==")
    # element.control.draw_outline()  # OK, too
    session.highlight(element.control, colour="green", thickness=2)
    print(element, element.control.get_properties())
    session.describe(element.control)

element

Locator pipeline (5 ops), success:
  [0] find_descendants(where={'automation_id': 'RootWebArea'}, depth=8, limit=1)  in=1  out=1  ok
  [1] child(index=0)  in=1  out=1  ok
  [2] child(index=0)  in=1  out=1  ok
  [3] find_descendants(where={'control_type': 'Button'}, depth=2, limit=100)  in=1  out=23  ok
  [4] take(index=9)  in=23  out=1  ok
element's children count = 0
 == result ==
PywinautoElement(control=<uia_controls.ButtonWrapper - 'Go Back', Button, -2926772732902020308>) {'class_name': 'ui-icon-button ui-jyslct ui-3nfvp2 ui-6s0dn4 ui-l56j7k ui-2lah0s ui-9f619 ui-exx8yu ui-1xpa7k ui-18d9i69 ui-1uhho1l ui-dj266r ui-1yf7rl7 ui-at24cr ui-j3b58b ui-c342km ui-ng3xce ui-jb2p0i ui-1qlqyl8 ui-1pd3egz ui-15bjb6t ui-1ypdohk ui-1s07b3s ui-ijokvz ui-1k57tk5 ui-784prv ui-1t137rt ui-9v5kkp ui-1uczgqu ui-1725o6r ui-1wfwxd8 ui-7s97pk ui-67bb7w ui-aqnwrm ui-ggy1nq ui-1oy24xj ui-g3p6pi ui-sagj69 ui-6ekqhr ui-1c071of ui-1tohbj2 ui-uupxpy ui-1eu7oa3 ui-1e94bgo ui-1043rbw ui-jbqb8w ui-1mpvj69 ui-1ooge

PywinautoElement(control=<uia_controls.ButtonWrapper - 'Go Back', Button, -2926772732902020308>)

In [ ]:
LocatorOp = ChildOp | FindDescendantsOp | FilterOp | TakeOp